# Generate AI restaurant reviews (OPEN models) — length-matched to real Yelp

Scales up + diversifies the AI-fake class for the generalized multi-modal Yelp model (Model B).
**Open models only** (no OpenAI API). Restaurant domain to match Yelp.

**Length-matched:** real Yelp restaurant reviews are short-to-medium (median ~75 words,
only ~22% over 150). If AI reviews were all long (~190w), the model would catch them on
LENGTH (a shortcut, AUC 0.81), not on writing style. So reviews are drawn from length
buckets weighted to match the real Yelp distribution.

Generators (ungated, native, distinct families): **Qwen2.5-7B · Llama-3-8B · Zephyr-7B · Yi-1.5-9B**
~2,500 each = ~10,000. `source` column tags the generator (for leave-one-generator-out).

### Instructions
1. **Runtime -> GPU** (H100/A100/G4; on T4 set `LOAD_4BIT=True`).
2. Run all cells (weights freed between models — disk won't fill).
3. Download `ai_restaurant_reviews_open.csv` -> put in project `data/` and tell the assistant.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import os, csv, random, gc, shutil
import torch
from transformers import pipeline, BitsAndBytesConfig

SEED = 42; random.seed(SEED)
MODELS = [("Qwen/Qwen2.5-7B-Instruct","qwen"),
          ("NousResearch/Meta-Llama-3-8B-Instruct","llama3"),
          ("HuggingFaceH4/zephyr-7b-beta","zephyr"),   # Mistral family
          ("01-ai/Yi-1.5-9B-Chat","yi")]               # Yi family (ungated, Llama-arch, native)
PER_MODEL = 2500          # 4 models x 2,500 ~ 10,000
BATCH     = 32            # H100/G4; lower to 16 on A100, 8 on T4
LOAD_4BIT = False         # True on T4
OUT       = "ai_restaurant_reviews_open.csv"

# length bucket -> (instruction, max_new_tokens)
LENGTH_BUCKETS = {
    "tiny":   ("in one very short sentence (a quick reaction, about 5-15 words)", 40),
    "short":  ("in one or two brief sentences (about 15-40 words)",              80),
    "medium": ("in a short paragraph (about 40-90 words)",                       170),
    "long":   ("in a detailed paragraph (about 120-200 words)",                  300),
}
# weights matched to REAL Yelp restaurant lengths (median ~75w, ~22% over 150w)
REST_BUCKETS = {"tiny":0.12, "short":0.25, "medium":0.35, "long":0.28}

PERSONAS = ["a busy parent","a college student","a retiree","a casual foodie","a hardcore food enthusiast",
            "a first-time visitor","a longtime regular","a skeptic","a bargain hunter","a critic at heart",
            "a tourist passing through","someone celebrating a special occasion","a picky eater","a vegan"]
STYLES   = ["polished and well-written","casual and conversational",
            "quick and informal, all lowercase with a couple of typos like a phone review",
            "enthusiastic and full of energy","measured, balanced and a little critical",
            "blunt and to the point","warm and storytelling"]
STARS    = [5,5,5,5,4,4,4,3,2,1,1]
SUBJECTS = ["an Italian restaurant","a sushi spot","a taco truck","a fine-dining steakhouse","a vegan cafe",
            "a burger joint","a Thai restaurant","a weekend brunch place","a pizzeria","a classic diner",
            "a ramen shop","a food truck","a Mexican cantina","an Indian restaurant","a Chinese restaurant",
            "a Korean BBQ place","a French bistro","a seafood restaurant","a BBQ smokehouse","a breakfast spot",
            "a coffee shop","a neighborhood bakery","a gastropub","a tapas bar","a Vietnamese pho place",
            "a Mediterranean restaurant","a deli","a fried chicken joint","an ice cream parlor","a wine bar"]

In [ ]:
def build_prompt(bucket):
    subject = random.choice(SUBJECTS); persona = random.choice(PERSONAS)
    style   = random.choice(STYLES);   stars = random.choice(STARS)
    instr   = LENGTH_BUCKETS[bucket][0]
    user = (f"Write a realistic online review of {subject} you visited, as if you are {persona}, "
            f"giving {stars} out of 5 stars. Write it {instr}, {style}. Invent a plausible specific "
            f"restaurant name and concrete details, and sound like a genuine human - "
            f"no header, no preamble, no star rating. Just output the review text.")
    return user, persona, style, stars, subject

In [ ]:
# resume-safe per-generator count
have = {fam: 0 for _, fam in MODELS}
if os.path.exists(OUT):
    import pandas as pd
    try:
        for s, c in pd.read_csv(OUT)["source"].value_counts().items():
            if s in have: have[s] = int(c)
    except Exception: pass
print("existing per generator:", have)

new = not os.path.exists(OUT)
f = open(OUT, "a", newline="", encoding="utf-8"); w = csv.writer(f)
if new: w.writerow(["text","domain","source","model_id","rating","persona","style","subject","length_bucket"])

for model_id, fam in MODELS:
    if have[fam] >= PER_MODEL:
        print(f"== {fam}: already has {have[fam]}, skipping =="); continue
    print(f"\n=== loading {model_id} ===")
    kw = dict(torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
    if LOAD_4BIT:
        kw = dict(model_kwargs={"quantization_config": BitsAndBytesConfig(
                      load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)},
                  device_map="auto", trust_remote_code=True)
    try:
        pipe = pipeline("text-generation", model=model_id, **kw)
    except Exception as e:
        print("  load failed, skipping:", e); continue
    tok = pipe.tokenizer
    if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id
    tok.padding_side = "left"

    made = 0
    for bucket, frac in REST_BUCKETS.items():        # group by bucket -> one max_new_tokens per batch
        count = round(PER_MODEL * frac)
        if count <= 0: continue
        max_tok = LENGTH_BUCKETS[bucket][1]
        prompts, meta = [], []
        for _ in range(count):
            user, persona, style, stars, subject = build_prompt(bucket)
            prompts.append([{"role":"user","content":user}]); meta.append((stars,persona,style,subject))
        for i in range(0, len(prompts), BATCH):
            batch = prompts[i:i+BATCH]
            try:
                outs = pipe(batch, max_new_tokens=max_tok, do_sample=True, temperature=0.9,
                            top_p=0.95, batch_size=len(batch), pad_token_id=tok.pad_token_id)
            except Exception as e:
                print("  gen error:", e); continue
            for j, o in enumerate(outs):
                rec = o[0] if isinstance(o, list) else o
                gen = rec["generated_text"]
                text = gen[-1]["content"] if isinstance(gen, list) else str(gen)
                text = text.strip().replace("\n", " ")
                stars, persona, style, subject = meta[i+j]
                if len(text.split()) >= 3:
                    w.writerow([text, "restaurants", fam, model_id, stars, persona, style, subject, bucket]); made += 1
        f.flush(); print(f"  {fam} {bucket}: total {made}")
    print(f"== {fam}: +{made} ==")
    del pipe; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--" +
                  model_id.replace("/", "--")), ignore_errors=True)
    print(f"   {fam} weights cache freed")

f.close()
print("done ->", OUT)

In [ ]:
import pandas as pd
df = pd.read_csv(OUT)
wl = df["text"].str.split().str.len()
print("total:", len(df), "| per generator:", df["source"].value_counts().to_dict())
print("median words:", int(wl.median()), "| %<=40w:", round((wl<=40).mean()*100), "% | %>150w:", round((wl>150).mean()*100),"%")
print("(target ~ real Yelp: median ~75, ~22% over 150w)")
from google.colab import files
files.download(OUT)